# ConvLSTM — Integrated Pipeline Walkthrough

This notebook walks through the integrated ConvLSTM pipeline on the `integrate` branch.

**Pipeline**: Shared config → load chunked data → train per-chunk model → evaluate multi-horizon → write metrics.

## 1. Setup & Imports

In [ ]:
import sys
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
import matplotlib.pyplot as plt

In [ ]:
# Add repo root to sys.path so we can import training.* modules
ROOT = Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Add ConvLSTM dir so we can import model.py directly
CONVLSTM_DIR = ROOT / "training" / "ConvLSTM"
if str(CONVLSTM_DIR) not in sys.path:
    sys.path.insert(0, str(CONVLSTM_DIR))

print(f"Repo root: {ROOT}")
print(f"ConvLSTM dir: {CONVLSTM_DIR}")

In [ ]:
# Now import all pipeline modules
from model import ConvLSTMPredictor
from training.common.config import load_config, resolve_path
from training.common.data import chunk_specs, load_chunk, model_matrix_to_convlstm_frames, ChunkSpec
from training.common.metrics import absolute_and_squared_errors_dbm, denormalize
from training.common.results import append_metric_rows, load_band_definitions, output_dir
from training.common.windowing import aligned_history_matrix, selected_horizon_index, target_rows_for

print("All imports OK")

## 2. Data Preparation

The integrated pipeline expects per-site CSV files in `evaluation/aerpaw/`. Our merged CSV is in `training/data/`. Let's extract the per-site columns into separate files if they don't exist.

In [ ]:
AERPAW_DIR = ROOT / "evaluation" / "aerpaw"
MERGED_CSV = ROOT / "training" / "data" / "merged_power_data_sub6GHz_avg_per_minute.csv"

AERPAW_DIR.mkdir(parents=True, exist_ok=True)
print(f"AERPAW target dir: {AERPAW_DIR}")
print(f"Merged CSV exists: {MERGED_CSV.exists()}")

In [ ]:
# Expected filenames by site
SITE_FILES = {
    "CC1": AERPAW_DIR / "ResultsCC1Feb2022_SigMF_power_1mhz_avg_per_minute.csv",
    "CC2": AERPAW_DIR / "ResultsCC2Feb2022_SigMF_power_1mhz_avg_per_minute.csv",
    "LW1": AERPAW_DIR / "ResultsLW1Feb2022_SigMF_power_1mhz_avg_per_minute.csv",
}

missing = [name for name, path in SITE_FILES.items() if not path.exists()]
if missing:
    print(f"Missing site CSVs: {missing}")
    print("Generating from merged CSV...")
    merged = np.loadtxt(MERGED_CSV, delimiter=",")
    print(f"Merged shape: {merged.shape}")  # (6839, 750)
    for i, name in enumerate(["CC1", "CC2", "LW1"]):
        site_data = merged[:, i * 250 : (i + 1) * 250]
        np.savetxt(SITE_FILES[name], site_data, delimiter=",", fmt="%.6f")
        print(f"  {name}: {SITE_FILES[name]} ({site_data.shape})")
else:
    print("All site CSVs already exist:")
    for name, path in SITE_FILES.items():
        print(f"  {name}: {path} ({np.loadtxt(path, delimiter=',').shape})")

## 3. Load & Inspect Shared Config

In [ ]:
config = load_config()

print("=== Data ===")
print(f"  data_dir: {config['data']['data_dir']}")
print(f"  reference_site: {config['data']['reference_site']}")
print(f"  test_splits: {config['data']['test_splits']}")
print(f"  chunks:")
for chunk in config['data']['chunks']:
    print(f"    {chunk['id']}: {chunk['start_mhz']}-{chunk['end_mhz']} MHz")

print(f"\n=== Windowing ===")
print(f"  lookback: {config['windowing']['lookback']}")
print(f"  horizons: {config['windowing']['horizons']}")
print(f"  min_history: {config['windowing']['min_history']}")

print(f"\n=== ConvLSTM ===")
ccfg = config['convlstm']
print(f"  input_sequence_length: {ccfg['input_sequence_length']}")
print(f"  prediction_horizon: {ccfg['prediction_horizon']}")
print(f"  batch_size: {ccfg['batch_size']}")
print(f"  epochs: {ccfg['epochs']}")
print(f"  learning_rate: {ccfg['learning_rate']}")
print(f"  model:")
for k, v in ccfg['model'].items():
    print(f"    {k}: {v}")

## 4. Explore a Single Chunk

Let's load one chunk's data and inspect what the pipeline produces.

In [ ]:
chunks = chunk_specs(config)
chunk = chunks[0]
print(f"Chunk: {chunk.chunk_id} ({chunk.start_mhz}-{chunk.end_mhz} MHz)")

data = load_chunk(config, chunk)

print(f"\nShared frequencies: {len(data.shared_frequencies)} bins")
print(f"  Range: {data.shared_frequencies[0]:.1f} - {data.shared_frequencies[-1]:.1f} MHz")
print(f"  Step: ~{data.shared_frequencies[1] - data.shared_frequencies[0]:.2f} MHz")

print(f"\nNormalization: {data.normalization}")

print(f"\nSplits:")
for split_name, split in data.splits.items():
    print(f"  {split_name}: model_input {split.model_input.shape}, raw_dbm {split.raw_dbm.shape}")

In [ ]:
# Visualize: raw dBm power over time for a few frequency bins
train_raw = data.splits["CC2_train"].raw_dbm

fig, axes = plt.subplots(2, 1, figsize=(14, 7))

# Pick 3 frequency bins to plot
n_bins = train_raw.shape[1]
bin_indices = [0, n_bins // 2, n_bins - 1]
bin_freqs = [data.shared_frequencies[i] for i in bin_indices]

ax = axes[0]
for idx, freq in zip(bin_indices, bin_freqs):
    ax.plot(train_raw[:, idx], label=f"{freq:.1f} MHz", linewidth=0.8)
ax.set_title(f"{chunk.chunk_id} CC2 Train — Raw dBm")
ax.set_xlabel("Time (minutes)")
ax.set_ylabel("Power (dBm)")
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1]
im = ax.imshow(train_raw.T, aspect="auto", cmap="viridis")
ax.set_title(f"{chunk.chunk_id} CC2 Train — Full Spectrogram")
ax.set_xlabel("Time (minutes)")
ax.set_ylabel("Frequency Bin")
plt.colorbar(im, ax=ax, label="Power (dBm)")

plt.tight_layout()
plt.show()

## 5. Training Windows

The pipeline creates sliding windows for training. Let's see how they work.

In [ ]:
# Reproduce the training window logic from train_integrated.py
train = data.splits["CC2_train"].model_input
lookback = int(config["convlstm"].get("input_sequence_length", config["windowing"]["lookback"]))
prediction_horizon = int(config["convlstm"].get("prediction_horizon", max(config["windowing"]["horizons"])))

frames = model_matrix_to_convlstm_frames(train)
origins = np.arange(lookback - 1, len(frames) - prediction_horizon, dtype=np.int64)

print(f"Train shape: {train.shape}")
print(f"Frames shape: {frames.shape}")
print(f"Number of training origins (windows): {len(origins)}")
print(f"Window: {lookback} input steps → {prediction_horizon} prediction steps")
print(f"First origin: {origins[0]}, Last origin: {origins[-1]}")
print(f"\nInput frame shape per sample: (1, {lookback}, 1, {train.shape[1]})")
print(f"Target shape per sample: ({prediction_horizon}, 1, 1, {train.shape[1]})")

In [ ]:
# Example: look at one training window
ex_origin = origins[0]
x = frames[ex_origin - lookback + 1 : ex_origin + 1]  # (lookback, 1, n_bins)
y = frames[ex_origin + 1 : ex_origin + prediction_horizon + 1]  # (60, 1, n_bins)

print(f"Input x: {x.shape}")
print(f"Target y: {y.shape}")

fig, axes = plt.subplots(2, 1, figsize=(12, 5))
ax = axes[0]
ax.imshow(x[:, 0, :].T, aspect="auto", cmap="viridis")
ax.set_title(f"Input — last {lookback} minutes (origin={ex_origin})")
ax.set_ylabel("Frequency bin")

ax = axes[1]
ax.imshow(y[:, 0, :].T, aspect="auto", cmap="viridis")
ax.set_title(f"Target — next {prediction_horizon} minutes")
ax.set_ylabel("Frequency bin")
ax.set_xlabel("Time step")
plt.tight_layout()
plt.show()

## 6. Build Model

Instantiate the ConvLSTMPredictor with the shared config.

In [ ]:
def build_model_config(config, n_bins):
    ccfg = config["convlstm"]
    return {
        "data": {"n_nodes": 1, "n_bins_per_node": n_bins, "node_names": ["CC2"]},
        "windowing": {
            "input_sequence_length": int(ccfg.get("input_sequence_length", config["windowing"]["lookback"])),
            "prediction_horizon": int(ccfg.get("prediction_horizon", max(config["windowing"]["horizons"]))),
        },
        "model": ccfg["model"],
    }

model_config = build_model_config(config, train.shape[1])
model = ConvLSTMPredictor(model_config)

total_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {total_params:,}")
print(f"Input: (B, {lookback}, 1, 1, {train.shape[1]})")
print(f"Output: (B, {prediction_horizon}, 1, 1, {train.shape[1]})")

# Forward pass check with dummy input
dummy = torch.randn(4, lookback, 1, 1, train.shape[1])
out = model(dummy)
print(f"\nDummy forward pass output shape: {out.shape}")

## 7. Train ConvLSTM (Single Chunk, Few Epochs)

Train one model on the first chunk. Halve epochs for a quick smoke test.

In [ ]:
# Quick smoke-test config: reduced epochs
smoke_config = {
    "convlstm": {
        "input_sequence_length": lookback,
        "prediction_horizon": prediction_horizon,
        "batch_size": 32,
        "epochs": 5,
        "learning_rate": 0.0002,
        "weight_decay": 0.004,
        "val_fraction": 0.1,
        "teacher_forcing_ratio": 1.0,
        "gradient_clip_norm": 5.0,
        "model": config["convlstm"]["model"],
    },
    "windowing": config["windowing"],
}

In [ ]:
# Create data loaders
val_fraction = 0.1
batch_size = 32
clip_norm = 5.0
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

val_count = max(1, int(len(origins) * val_fraction))
train_origins = origins[:-val_count]
val_origins = origins[-val_count:]

print(f"Train windows: {len(train_origins)}, Val windows: {len(val_origins)}")

train_loader = DataLoader(
    ConvLSTMWindowDataset(frames, lookback, prediction_horizon, train_origins),
    batch_size=batch_size, shuffle=True, drop_last=True,
)
val_loader = DataLoader(
    ConvLSTMWindowDataset(frames, lookback, prediction_horizon, val_origins),
    batch_size=batch_size, shuffle=False,
)

In [ ]:
# Define the ConvLSTMWindowDataset locally (used above)
class ConvLSTMWindowDataset(Dataset):
    def __init__(self, frames: np.ndarray, lookback: int, prediction_horizon: int, origins: np.ndarray):
        self.frames = torch.from_numpy(frames).float()
        self.lookback = lookback
        self.prediction_horizon = prediction_horizon
        self.origins = origins.astype(np.int64)

    def __len__(self) -> int:
        return len(self.origins)

    def __getitem__(self, idx: int):
        origin = int(self.origins[idx])
        x = self.frames[origin - self.lookback + 1 : origin + 1].unsqueeze(1)
        y = self.frames[origin + 1 : origin + self.prediction_horizon + 1].unsqueeze(1)
        return x, y

# Recreate dataloaders with the locally defined class
train_loader = DataLoader(
    ConvLSTMWindowDataset(frames, lookback, prediction_horizon, train_origins),
    batch_size=batch_size, shuffle=True, drop_last=True,
)
val_loader = DataLoader(
    ConvLSTMWindowDataset(frames, lookback, prediction_horizon, val_origins),
    batch_size=batch_size, shuffle=False,
)
print("DataLoaders ready")

In [ ]:
# Training loop
model = ConvLSTMPredictor(model_config).to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=smoke_config["convlstm"]["learning_rate"],
    weight_decay=smoke_config["convlstm"]["weight_decay"],
)

epochs = smoke_config["convlstm"]["epochs"]
teacher_forcing_ratio = smoke_config["convlstm"]["teacher_forcing_ratio"]

train_losses = []
val_losses = []
best_loss = float("inf")

for epoch in range(1, epochs + 1):
    model.train()
    train_loss = 0.0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        pred = model(x, y_teacher=y, teacher_forcing_ratio=teacher_forcing_ratio)
        loss = criterion(pred, y)
        loss.backward()
        if clip_norm > 0:
            nn.utils.clip_grad_norm_(model.parameters(), clip_norm)
        optimizer.step()
        train_loss += loss.item() * x.size(0)
    train_loss /= max(len(train_loader.dataset), 1)
    train_losses.append(train_loss)

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            pred = model(x)
            val_loss += criterion(pred, y).item() * x.size(0)
    val_loss /= max(len(val_loader.dataset), 1)
    val_losses.append(val_loss)

    print(f"Epoch {epoch:03d}/{epochs} train={train_loss:.6f} val={val_loss:.6f}")

    if val_loss < best_loss:
        best_loss = val_loss

In [ ]:
# Plot training curve
plt.figure(figsize=(10, 4))
plt.plot(train_losses, label="Train MSE", marker=".")
plt.plot(val_losses, label="Val MSE", marker=".")
plt.xlabel("Epoch")
plt.ylabel("MSE")
plt.title(f"ConvLSTM Training — {chunk.chunk_id} ({5} epochs)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()
print(f"Best val loss: {best_loss:.6f}")

## 8. Evaluate on Test Set

Use the trained model to predict test targets and compute per-horizon metrics.

In [ ]:
# Collect test split data
split = data.splits["CC2_test"]
full_x = np.vstack([train, split.model_input]).astype(np.float32)
full_raw = np.vstack([data.splits["CC2_train"].raw_dbm, split.raw_dbm]).astype(np.float32)
history_offset = len(train)
min_history = int(config["windowing"].get("min_history", 4320))
horizons = [int(h) for h in config["windowing"]["horizons"]]

print(f"Full array shape: {full_x.shape}")
print(f"Test rows: {len(split.raw_dbm)} (offset={history_offset})")
print(f"Horizons to evaluate: {horizons}")

In [ ]:
# Predict for each horizon
predictions = {}
for horizon in horizons:
    target_rows = target_rows_for(len(split.raw_dbm), history_offset, horizon, lookback, min_history)
    origins = target_rows - horizon
    histories = aligned_history_matrix(full_x, origins, horizon=0, lookback=lookback)
    histories = histories[:, :, None, None, :].astype(np.float32)

    loader = DataLoader(torch.from_numpy(histories).float(), batch_size=batch_size, shuffle=False)
    preds = []
    model.eval()
    with torch.no_grad():
        for x_batch in loader:
            out = model(x_batch.to(device))
            preds.append(out[:, selected_horizon_index(horizon), 0, 0, :].cpu().numpy())
    predictions[horizon] = np.concatenate(preds, axis=0).astype(np.float32)
    target = full_raw[target_rows]
    pred_dbm, abs_err, sq_err = absolute_and_squared_errors_dbm(predictions[horizon], target, data.normalization)
    print(f"H={horizon:2d}: n_targets={len(target_rows):4d}, MAE={np.mean(abs_err):.4f} dB, RMSE={np.sqrt(np.mean(sq_err)):.4f} dB")

In [ ]:
# Visualize predictions vs ground truth for a single frequency bin
bin_idx = n_bins // 2  # middle frequency
freq = data.shared_frequencies[bin_idx]

fig, axes = plt.subplots(len(horizons), 1, figsize=(12, 3 * len(horizons)), sharex=True)

for ax, horizon in zip(axes, horizons):
    target_rows = target_rows_for(len(split.raw_dbm), history_offset, horizon, lookback, min_history)
    subset = slice(0, min(200, len(target_rows)))
    t = target_rows[subset] - history_offset  # relative time in test set

    gt = split.raw_dbm[target_rows[subset] - history_offset, bin_idx]
    pred_dbm, _, _ = absolute_and_squared_errors_dbm(
        predictions[horizon][subset, bin_idx:bin_idx+1],
        full_raw[target_rows[subset], bin_idx:bin_idx+1],
        data.normalization,
    )
    ax.plot(t, gt, label="Ground Truth", alpha=0.8, linewidth=0.8)
    ax.plot(t, pred_dbm[:, 0], label="Prediction", alpha=0.8, linewidth=0.8)
    ax.set_ylabel("Power (dBm)")
    ax.set_title(f"Horizon={horizon} min — Bin {freq:.1f} MHz")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel("Test time (minutes)")
plt.tight_layout()
plt.show()

## 9. Full Pipeline Run

Run the complete integrated training + evaluation for all chunks. This saves metrics to `training/results/ConvLSTM/`.

In [ ]:
from training.ConvLSTM.train_integrated import evaluate_chunk as convlstm_evaluate_chunk
from training.common.results import output_dir

out = output_dir(config, "ConvLSTM")
(out / "models").mkdir(parents=True, exist_ok=True)
bands = load_band_definitions(config)

all_aggregate = []
all_frequency = []
all_band = []

for chunk in chunk_specs(config):
    print(f"\n{'='*60}")
    print(f"Processing {chunk.chunk_id} ({chunk.start_mhz}-{chunk.end_mhz} MHz)")
    print(f"{'='*60}")
    agg, freq, band = convlstm_evaluate_chunk(config, chunk, bands, out)
    all_aggregate.extend(agg)
    all_frequency.extend(freq)
    all_band.extend(band)
    print(f"  → {len(agg)} aggregate rows")

pd.DataFrame(all_aggregate).to_csv(out / "aggregate_metrics.csv", index=False)
pd.DataFrame(all_frequency).to_csv(out / "per_frequency_metrics.csv", index=False)
pd.DataFrame(all_band).to_csv(out / "per_band_metrics.csv", index=False)
print(f"\nWrote {len(all_aggregate)} total rows to {out / 'aggregate_metrics.csv'}")

In [ ]:
# Display the results table
results = pd.read_csv(out / "aggregate_metrics.csv")
results

## 10. Assemble Overall Results

Combine with baselines and other model outputs.

In [ ]:
from training.common.assemble_results import combine_metric_files, add_skill_scores, write_summary

input_dirs = [
    resolve_path(path)
    for path in ["training/results/baselines", "training/results/ConvLSTM"]
    if resolve_path(path).exists()
]

if input_dirs:
    aggregate = add_skill_scores(combine_metric_files(input_dirs, "aggregate_metrics.csv"))
    print(f"Combined aggregate: {len(aggregate)} rows from {len(input_dirs)} input dirs")
    display(aggregate)
else:
    print("No result directories found yet. Run baselines first:")
    print("  python3 evaluation/scripts/run_spectrum_steps_5_6.py --normalize --output-dir training/results/baselines")